In [120]:
import pandas as pd
import geopandas as gpd
import re

excel_path = "MTSP-Data-FY25.xlsx"
shp_path   = "townssurvey_shp/TOWNSSURVEY_POLY.shp"

df  = pd.read_excel(excel_path)
gdf = gpd.read_file(shp_path)

In [121]:
mask  = df["state_name"].astype(str).str.strip().str.upper() == "MASSACHUSETTS"
df_ma = df.loc[mask].copy()

# remove "town"/"city"
df_ma["county_town_name"] = (
    df_ma["county_town_name"]
      .astype(str)
      .str.replace("\u00A0", " ", regex=False)           # NBSP -> space
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.replace(r"\b(town|city)\b(\s+\b(town|city)\b)*\s*$", "",
                   flags=re.IGNORECASE, regex=True)
      .str.strip()
)

# county_town_name
total_ctn  = df_ma["county_town_name"].count()
unique_ctn = df_ma["county_town_name"].nunique()
print(f"county_town_name: {total_ctn} values, including {unique_ctn} unique values")

county_town_name: 351 values, including 351 unique values


In [122]:
# shapefile
print(f"Number of town or city: {len(gdf)}")
print()
print(gdf.columns)
print()
print(gdf[["TOWN", "TOWN_ID"]].head())

Number of town or city: 1239

Index(['TOWN', 'TOWN_ID', 'TYPE', 'COUNTY', 'FIPS_STCO', 'FOURCOLOR', 'AREA_ACRES', 'AREA_SQMI', 'ISLAND', 'COASTAL_PO', 'SHAPE_Leng', 'SHAPE_Area', 'geometry'], dtype='object')

               TOWN  TOWN_ID
0           EASTHAM       86
1           EASTHAM       86
2           EASTHAM       86
3           DUXBURY       82
4  EAST BRIDGEWATER       83


In [123]:
# compare
def _norm_series_lower(s: pd.Series) -> set[str]:
    return set(
        s.astype(str)
         .str.replace("\u00A0", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
         .str.lower()
         .unique()
    )

set_excel = _norm_series_lower(df_ma["county_town_name"])
set_shp   = _norm_series_lower(gdf["TOWN"])

common         = sorted(set_excel & set_shp)
only_in_excel  = sorted(set_excel - set_shp)
only_in_shp    = sorted(set_shp   - set_excel)

In [124]:
print("Comparison Results (case-insensitive)")
print(f"Unique town names in Excel: {len(set_excel)}")
print(f"Unique town names in Shapefile: {len(set_shp)}")
print(f"Number of matches: {len(common)}")
print(f"Number of towns only in Excel: {len(only_in_excel)}")
print(f"Number of towns only in Shapefile: {len(only_in_shp)}")

Comparison Results (case-insensitive)
Unique town names in Excel: 351
Unique town names in Shapefile: 351
Number of matches: 351
Number of towns only in Excel: 0
Number of towns only in Shapefile: 0


In [125]:
df = pd.read_csv("CHAPA_applications_concat_unclean.csv")

US_STATES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA",
    "ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC","PR"
}

In [126]:
direction_map = {
    r'\bn\.?\b': 'north',
    r'\bs\.?\b': 'south',
    r'\be\.?\b': 'east',
    r'\bw\.?\b': 'west'
}

for abbr, full in direction_map.items():
    df['Current Residence'] = df['Current Residence'].str.replace(abbr, full, regex=True, case=False)

In [127]:
with_dot = [v for v in df["Current Residence"].dropna() if "." in str(v)]
print("value with_dot")
print(with_dot)

value with_dot
['north. Attleboro', 'north. Andover', 'north. Billerica', 'Lowell, MA. ', 'south. Hamilton', 'Waltham. Massachussettes', "Englewood FL (health issues need to be closer to family and can't afford anything so this would be perfect."]


In [128]:
df['Current Residence'] = df['Current Residence'].str.replace(r'\d+', '', regex=True)

pattern = r'\s(' + '|'.join(sorted(US_STATES)) + r')\b'

extracted = (
    df['Current Residence']
    .str.extract(pattern, flags=re.IGNORECASE)[0]
    .str.upper()
)

if 'Current Residence State' in df.columns:
    df['Current Residence State'] = df['Current Residence State'].combine_first(extracted)
else:
    df['Current Residence State'] = extracted

df['Current Residence'] = (
    df['Current Residence']
    .str.replace(pattern, '', flags=re.IGNORECASE, regex=True)
    .str.replace(r'\s{2,}', ' ', regex=True)
    .str.strip()
)


In [117]:
print(f"Number of rows where the state abbreviation has been filled in: {df['Current Residence State'].notna().sum()}")

Number of rows where the state abbreviation has been filled in: 136


In [130]:
print(df['Current Residence State'].unique())

print("Number of unique states:", df['Current Residence State'].nunique(dropna=True))

[nan 'NH' 'FL' 'MA' 'RI' 'CT' 'AR' 'SC' 'ME']
Number of unique states: 8


In [118]:
with_comma = [v for v in df["Current Residence"].dropna() if "," in str(v)]
print("with_comma")
print(with_comma)

with_comma
['Nashua,NH', 'Nashua,', 'Cape Coral,', 'Chester,', 'North Reading,', 'Nashua,', 'Hampstead,', 'Lowell,MA', 'Kensington,', 'Plaistow,', 'Nashua,', 'Nashua,', 'Providence,', 'Haverhill,', 'Winthrop,', 'Georgetown,', 'Lowell,.', 'Tewksbury,', 'Haverhill,', 'Salisbury,', 'Worcester,', 'Auburn,', 'Boston,', 'Bridgewater,', 'Shirley,', 'Cranston,', 'Taunton,', 'Malden,', 'Ayer,', 'Marlborough,', 'Nashua,', 'Cambridge,', 'Quincy,', 'Westwood,', 'Dorchester,', 'Dorchester,', 'Marlborough,MA', 'Foxboro,', 'Mansfield,', 'West Newbury,', 'Kensington,', 'Byfield,', 'Oak Bluffs,', 'Edgartown,', 'Wellesley,', 'Taunton,', 'Taunton,', 'Pawtucket,', 'Marlborough,', 'Clinton,', 'Lowell,', 'Dracut,', 'Litchfield,', 'Waltham , US', 'Clinton,', 'Rutland,Ma', 'Francestown,', 'Marlborough,MA', 'Hollis,', 'Wellesley,', 'Brockton,', 'LOWELL,', 'Upton,MA', 'Littleton,', 'Andover,', 'Norwood ,', 'Littleton,', 'Dracut,', 'hudson,', 'Milford,', 'Hingham, Massachusetts', 'Norwood,', 'Mendon,', 'Fort Smi

In [119]:
special_chars = set()

for val in df["Current Residence"].dropna().astype(str):
    chars = re.findall(r"[^A-Za-z0-9\s]", val)
    special_chars.update(chars)


print("Unique special characters found in 'Current Residence':")
print(sorted(special_chars))
print(f"\nTotal unique special characters: {len(special_chars)}")

Unique special characters found in 'Current Residence':
["'", '(', ',', '-', '.', '/']

Total unique special characters: 6


In [84]:
df["Current Residence"] = df["Current Residence"].replace(r"[\'\(\)\-\.\/]", " ", regex=True)

In [85]:
after = (
    df["Current Residence"]
    .dropna()
    .astype(str)
    .loc[lambda s: s.str.contains(",")]
    .str.split(",", n=1).str[1]
    .str.strip()
    .str.upper()
)

uniq_after = sorted(set(after))

def classify(x: str) -> str:
    if len(x) == 2 and x.isalpha():
        return "US_STATE_ABBR" if x in US_STATES else "TWO_LETTER_NON_STATE"
    else:
        return "OTHER"
        
grouped = {"US_STATE_ABBR": [], "TWO_LETTER_NON_STATE": [], "OTHER": []}
for v in uniq_after:
    grouped[classify(v)].append(v)

In [86]:
# 7) Print results
print(f"After-comma (trimmed) unique values: {len(uniq_after)}")
print(f"- Valid U.S. state abbreviations: {len(grouped['US_STATE_ABBR'])}")
print(f"- Two-letter but non-state abbreviations: {len(grouped['TWO_LETTER_NON_STATE'])}")
print(f"- Other types: {len(grouped['OTHER'])}")

print("\nValid U.S. State Abbreviations")
print(sorted(grouped["US_STATE_ABBR"]))

print("\nTwo-Letter but Non-State Abbreviations")
print(sorted(grouped["TWO_LETTER_NON_STATE"]))

print("\nOther")
print(sorted(grouped["OTHER"]))

After-comma (trimmed) unique values: 7
- Valid U.S. state abbreviations: 2
- Two-letter but non-state abbreviations: 1
- Other types: 4

Valid U.S. State Abbreviations
['MA', 'NH']

Two-Letter but Non-State Abbreviations
['US']

Other
['', 'MASS', 'MASSACHUSETTS', 'NEW HAMPSHIRE']


In [87]:
state_map = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "FLORIDA": "FL", "GEORGIA": "GA",
    "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS",
    "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD", "MASSACHUSETTS": "MA",
    "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT",
    "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM", "NEW YORK": "NY", "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND",
    "OHIO": "OH", "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI",
    "SOUTH CAROLINA": "SC", "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX",
    "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI", "WYOMING": "WY", "DISTRICT OF COLUMBIA": "DC"
}
us_states = set(state_map.values())

if 'Current Residence State' not in df.columns:
    df['Current Residence State'] = None

for idx, val in df["Current Residence"].items():
    if isinstance(val, str) and "," in val:
        if pd.notna(df.at[idx, "Current Residence State"]):
            df.at[idx, "Current Residence"] = val.replace(",", "").strip()
            continue

        before, after = val.split(",", 1)
        after_clean = after.strip().upper()
        state = None

        if len(after_clean) == 2 and after_clean in us_states:
            state = after_clean
        elif after_clean in state_map:
            state = state_map[after_clean]
        elif "MA" in after_clean:
            state = "MA"
        elif "NEW HAMPSHIRE" in after_clean:
            state = "NH"
        else:
            print(f"[Unmatched] Row {idx}: {val}")

        df.at[idx, "Current Residence State"] = state
        df.at[idx, "Current Residence"] = before.strip()

In [88]:
remove_ma = ["massachusetts", "mass", "ma", "ma."]
pattern_ma = re.compile(r'\b(?:' + '|'.join(remove_ma) + r')\b', flags=re.IGNORECASE)

if "Current Residence State" not in df.columns:
    df["Current Residence State"] = None

for idx, val in df["Current Residence"].items():
    if isinstance(val, str):
        if pattern_ma.search(val):
            df.at[idx, "Current Residence State"] = "MA"

            cleaned = pattern_ma.sub("", val)
            cleaned = re.sub(r'\s+', ' ', cleaned).strip()
            df.at[idx, "Current Residence"] = cleaned

In [89]:
print("Number of matched state:", df['Current Residence State'].notna().sum())

Number of matched state: 154


In [90]:
df2 = pd.read_excel(excel_path)

target_states = {"MASSACHUSETTS", "RHODE ISLAND", "NEW HAMPSHIRE"}
df2['state_name'] = df2['state_name'].astype(str).str.upper().str.strip()
df2 = df2[df2['state_name'].isin(target_states)].copy()

def clean_ctn(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.replace("\u00A0", " ", regex=False)            # NBSP -> space
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
         .str.replace(r"\b(town|city)\b(\s+\b(town|city)\b)*\s*$", "",
                      flags=re.IGNORECASE, regex=True)
         .str.strip()
    )

df2_ma = df2[df2['state_name'].eq('MASSACHUSETTS')].copy()
df2_ri = df2[df2['state_name'].eq('RHODE ISLAND')].copy()
df2_nh = df2[df2['state_name'].eq('NEW HAMPSHIRE')].copy()

for _df2 in (df2_ma, df2_ri, df2_nh):
    _df2['county_town_name'] = clean_ctn(_df2['county_town_name'])

ma_cities = df2_ma['county_town_name'].dropna().tolist()
ri_cities = df2_ri['county_town_name'].dropna().tolist()
nh_cities = df2_nh['county_town_name'].dropna().tolist()

ma_list = sorted(set([str(x).strip().lower() for x in ma_cities if str(x).strip()]))
ri_list = sorted(set([str(x).strip().lower() for x in ri_cities if str(x).strip()]))
nh_list = sorted(set([str(x).strip().lower() for x in nh_cities if str(x).strip()]))

print(f"MA unique towns: {len(ma_list)}")
print(f"RI unique towns: {len(ri_list)}")
print(f"NH unique towns: {len(nh_list)}")

MA unique towns: 351
RI unique towns: 39
NH unique towns: 259


In [91]:
df2.columns

Index(['fips', 'stusps', 'state', 'state_name', 'hud_area_code',
       'hud_area_name', 'county', 'County_Name', 'county_town_name', 'metro',
       'median2025', 'lim50_25p1', 'lim50_25p2', 'lim50_25p3', 'lim50_25p4',
       'lim50_25p5', 'lim50_25p6', 'lim50_25p7', 'lim50_25p8', 'Lim60_25p1',
       'Lim60_25p2', 'Lim60_25p3', 'Lim60_25p4', 'Lim60_25p5', 'Lim60_25p6',
       'Lim60_25p7', 'Lim60_25p8', 'HERA_Lim_type25', 'Lim50_HERA_25p1',
       'Lim50_HERA_25p2', 'Lim50_HERA_25p3', 'Lim50_HERA_25p4',
       'Lim50_HERA_25p5', 'Lim50_HERA_25p6', 'Lim50_HERA_25p7',
       'Lim50_HERA_25p8', 'Lim60_HERA_25p1', 'Lim60_HERA_25p2',
       'Lim60_HERA_25p3', 'Lim60_HERA_25p4', 'Lim60_HERA_25p5',
       'Lim60_HERA_25p6', 'Lim60_HERA_25p7', 'Lim60_HERA_25p8'],
      dtype='object')

In [92]:
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein as lev

SCORER = fuzz.WRatio
NEAR_EXACT_MATCH = 99
REVIEW_MATCH = 85

SRC_COL = "Current Residence"
assert SRC_COL in df.columns, f"Column {SRC_COL} does not exist"

ma_set, ri_set, nh_set = set(ma_list), set(ri_list), set(nh_list)

df["Current Address"] = df[SRC_COL]

def normalize_for_match(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

df["normalized_address"] = df["Current Address"].map(normalize_for_match)

def _best_among_states(s):
    best = (None, -1, None)
    for state, city_list in [("MA", ma_list), ("RI", ri_list), ("NH", nh_list)]:
        if not city_list:
            continue
        cand = process.extractOne(s, city_list, scorer=SCORER)
        if cand:
            c, sc, _ = cand
            if sc > best[1]:
                best = (c, sc, state)
    return best

def classify_match(s_raw):
    s = "" if pd.isna(s_raw) else str(s_raw).strip().lower()
    if not s:
        return ("", "", 0, "empty")

    if s in ma_set:
        return (s, "MA", 100, "EXACT_MATCH")
    if s in ri_set:
        return (s, "RI", 100, "EXACT_MATCH")
    if s in nh_set:
        return (s, "NH", 100, "EXACT_MATCH")

    cand, score, state = _best_among_states(s)
    if not cand:
        return (s, "", 0, "no_match")

    dist = lev.distance(s, cand)

    if dist == 1 and len(cand) == len(s) + 1:
        return (cand, state, int(score), "A_missing_one_letter")
    elif dist == 1 and len(cand) + 1 == len(s):
        return (cand, state, int(score), "B_extra_one_letter")
    elif dist <= 2:
        return (cand, state, int(score), "C_similar_but_different_city")
    elif score >= NEAR_EXACT_MATCH:
        return (cand, state, int(score), "NEAR_EXACT_MATCH")
    elif score >= REVIEW_MATCH:
        return (cand, state, int(score), "REVIEW_MATCH")
    else:
        return (cand, state, int(score), "no_match")

fuzz_output = df["normalized_address"].apply(classify_match)
df[["matched_city", "matched_state", "match_score", "match_type"]] = (
    pd.DataFrame(fuzz_output.tolist(), index=df.index)
)

if "Current Residence State" not in df.columns:
    df["Current Residence State"] = pd.Series(pd.NA, index=df.index, dtype="string")

mask_fill = df["match_type"].isin(["EXACT_MATCH"]) & df["matched_state"].notna()

for idx, row in df.loc[mask_fill].iterrows():
    new_state = row["matched_state"]
    old_state = row["Current Residence State"]

    if pd.isna(old_state) or old_state == "":
        df.at[idx, "Current Residence State"] = new_state
    elif str(old_state).strip().upper() != str(new_state).strip().upper():
        print(f"[Conflict] Row {idx}: DIFFERENT 'STATE'")
        print(row[["Current Address", "matched_city", "matched_state", "Current Residence State", "match_score", "match_type"]])
        print("-" * 80)

print(f"Filled {mask_fill.sum()} rows with MA/RI/NH state names (conflicts have been printed)")

print("\nMatch type distribution:")
print(df["match_type"].value_counts())

not_exact = df["match_type"].ne("EXACT_MATCH")
review_view = (
    df.loc[not_exact, ["Current Address", "matched_city", "match_type"]]
      .reset_index(drop=True).reset_index(names="row_id")
)
print(f"\nNot EXACT_MATCH row number：{len(review_view)}")
print(review_view.head(30).to_string(index=False))

[Conflict] Row 48: DIFFERENT 'STATE'
Current Address                Chester
matched_city                   chester
matched_state                       MA
Current Residence State             NH
match_score                        100
match_type                 EXACT_MATCH
Name: 48, dtype: object
--------------------------------------------------------------------------------
[Conflict] Row 508: DIFFERENT 'STATE'
Current Address            Marlborough
matched_city               marlborough
matched_state                       MA
Current Residence State             CT
match_score                        100
match_type                 EXACT_MATCH
Name: 508, dtype: object
--------------------------------------------------------------------------------
[Conflict] Row 570: DIFFERENT 'STATE'
Current Address             Dorchester
matched_city                dorchester
matched_state                       NH
Current Residence State             MA
match_score                        100
match_type   

In [93]:
import pandas as pd

# Initialization: add unique index + result columns
def init_review(df: pd.DataFrame):
    # Sequential unique index (starting from 1)
    df['u_index'] = pd.RangeIndex(1, len(df) + 1, name='u_index')
    # Result columns (keep existing ones if present, do not overwrite)
    if 'matched_city_confirmed' not in df.columns:
        df['matched_city_confirmed'] = pd.NA
    if 'match_confirm' not in df.columns:
        df['match_confirm'] = pd.NA

    # EXACT: automatic confirmation
    exact_mask = df['match_type'].eq('EXACT_MATCH')
    df.loc[exact_mask & df['matched_city'].notna(), 'matched_city_confirmed'] = df.loc[exact_mask, 'matched_city']
    df.loc[exact_mask, 'match_confirm'] = 'automatically confirmed'

    print(f"Initialization complete: {int(exact_mask.sum())} rows automatically confirmed as EXACT_MATCH.")

    return df

# View a specific entry (by u_index)
def show_item(df: pd.DataFrame, u_index: int):
    r = df.loc[df['u_index'].eq(u_index)]
    if r.empty:
        print(f"No row found with u_index={u_index}.")
        return
    cols = ['u_index', 'Current Address', 'matched_city', 'match_type', 'match_score',
            'matched_city_confirmed', 'match_confirm']
    cols = [c for c in cols if c in df.columns]
    print(r[cols].to_string(index=False))

# Action: approve auto match (manually)
def accept_match(df: pd.DataFrame, u_index: int):
    mask = df['u_index'].eq(u_index)
    if not mask.any():
        print(f"No row found with u_index={u_index}.")
        return df
    df.loc[mask, 'matched_city_confirmed'] = df.loc[mask, 'matched_city']
    df.loc[mask, 'match_confirm'] = 'manual confirmed'
    print(f"u_index={u_index} manually confirmed (with auto matched).")
    return df.loc[mask]

def override_city(df: pd.DataFrame, u_index: int, new_city: str):
    mask = df['u_index'].eq(u_index)
    if not mask.any():
        print(f"No row found with u_index={u_index}.")
        return df
    df.loc[mask, 'matched_city_confirmed'] = new_city
    df.loc[mask, 'match_confirm'] = 'manual confirmed'
    print(f"u_index={u_index} manually modified city to '{new_city}' and confirmed.")
    return df.loc[mask]

def mark_empty(df: pd.DataFrame, u_index: int):
    mask = df['u_index'].eq(u_index)
    if not mask.any():
        print(f"No row found with u_index={u_index}.")
        return df
    df.loc[mask, 'matched_city_confirmed'] = pd.NA
    df.loc[mask, 'match_confirm'] = 'empty'
    print(f"u_index={u_index} marked as empty.")
    return df.loc[mask]

def mark_not_found(df: pd.DataFrame, u_index: int):
    mask = df['u_index'].eq(u_index)
    if not mask.any():
        print(f"No row found with u_index={u_index}.")
        return df
    df.loc[mask, 'matched_city_confirmed'] = pd.NA
    df.loc[mask, 'match_confirm'] = 'not found'
    print(f"u_index={u_index} marked as not found.")
    return df.loc[mask]

def next_pending(df: pd.DataFrame):
    mask = df['match_type'].ne('EXACT_MATCH') & df['match_confirm'].isna()
    nxt = df.loc[mask, 'u_index'].min()
    if pd.isna(nxt):
        print("DONE!!!!!! No remaining non-EXACT rows to process.")
        return None
    print(f"Next pending u_index = {int(nxt)}")
    return int(nxt)

In [94]:
def mark_boston(df: pd.DataFrame, u_index: int):
    mask = df['u_index'].eq(u_index)
    if not mask.any():
        print(f"No row found with u_index={u_index}.")
        return df
    df.loc[mask, 'matched_city_confirmed']  = 'boston'
    df.loc[mask, 'matched_state_confirmed'] = 'MA'
    df.loc[mask, 'match_confirm']           = 'manual confirmed'
    print(f"u_index={u_index} manually set to 'boston, MA' and confirmed.")
    return df.loc[mask]

In [44]:
# One-time initialization (run only once)
init_review(df)

Initialization complete: 1574 rows automatically confirmed as EXACT_MATCH.


,Unnamed: 0,ID Number,Submission Date,Application Property,Source,Age,Race/Ethnicity,Disability,Current Residence,HH Size,...,Current Residence State,Current Address,normalized_address,matched_city,matched_state,match_score,match_type,u_index,matched_city_confirmed,match_confirm
0,0,150733,2022-02-08 00:00:00.000,8 Ciderpress Way,NaN,65,white,NaN,somerville,1,...,MA,somerville,somerville,somerville,MA,100,EXACT_MATCH,1,somerville,automatically confirmed
1,1,439181,2022-02-08 00:00:00.000,8 Ciderpress Way,NaN,59,hispanic,NaN,westford,2,...,MA,westford,westford,westford,MA,100,EXACT_MATCH,2,westford,automatically confirmed
2,2,697399,2022-02-08 00:00:00.000,8 Ciderpress Way,NaN,NaN,black/afr am,NaN,malden,2,...,MA,malden,malden,malden,MA,100,EXACT_MATCH,3,malden,automatically confirmed
3,3,191565,2022-02-08 00:00:00.000,8 Ciderpress Way,NaN,NaN,white,NaN,haverhill,1,...,MA,haverhill,haverhill,haverhill,MA,100,EXACT_MATCH,4,haverhill,automatically confirmed
4,4,226436,2022-02-08 00:00:00.000,8 Ciderpress Way,NaN,60,white,NaN,Nashua,1,...,NH,Nashua,nashua,nashua,NH,100,EXACT_MATCH,5,nashua,automatically confirmed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1735,1355,583613,2025-01-14 17:05:55.000,72 Buttercup Lane ~ Grafton,Real Estate Agent,36.0,Middle Eastern or North African,No,Worcester,4,...,MA,Worcester,worcester,worcester,MA,100,EXACT_MATCH,1736,worcester,automatically confirmed
1736,1356,575544,2025-01-12 15:11:57.000,72 Buttercup Lane ~ Grafton,Zillow/Trulia/Other Website,25.0,White,No,Sutton,1,...,MA,Sutton,sutton,sutton,MA,100,EXACT_MATCH,1737,sutton,automatically confirmed
1737,1357,954465,2025-01-11 23:11:20.000,72 Buttercup Lane ~ Grafton,MyMassHome,62.0,White,No,northborough,5,...,MA,northborough,northborough,northborough,MA,100,EXACT_MATCH,1738,northborough,automatically confirmed
1738,1358,218300,2025-01-02 13:32:05.000,"250 Main St, Unit 410 ~ Hudson (age restricted...",Zillow/Trulia/Other Website,60.0,White,Yes,Hudson,0,...,MA,Hudson,hudson,hudson,MA,100,EXACT_MATCH,1739,hudson,automatically confirmed


In [107]:
# Test if in a list
"grafton" in ma_list

True

In [443]:
# Find the next pending item
uid = next_pending(df)
show_item(df, uid)
df.loc[df["u_index"] == uid, ["Current Address", "matched_city"]]

DONE!!!!!! No remaining non-EXACT rows to process.
No row found with u_index=None.


,Current Address,matched_city


In [442]:
accept_match(df, uid)

u_index=1733 manually confirmed (with auto matched).


,Unnamed: 0,ID Number,Submission Date,Application Property,Source,Age,Race/Ethnicity,Disability,Current Residence,HH Size,...,normalized_address,matched_city,matched_state,match_score,match_type,u_index,matched_city_confirmed,match_confirm,matched_state_confirmed,matched_county_confirmed
1732,1352,204988,2025-01-15 17:34:10.000,72 Buttercup Lane ~ Grafton,Real Estate Agent,44.0,Asian,No,North Attleboro,4,...,north attleboro,north attleborough,MA,90,REVIEW_MATCH,1733,north attleborough,manual confirmed,NaN,<NA>


In [349]:
mark_empty(df, uid)

u_index=1208 marked as empty.


,Unnamed: 0,ID Number,Submission Date,Application Property,Source,Age,Race/Ethnicity,Disability,Current Residence,HH Size,...,normalized_address,matched_city,matched_state,match_score,match_type,u_index,matched_city_confirmed,match_confirm,matched_state_confirmed,matched_county_confirmed
1207,827,478470,2024-08-07 15:08:55.000,8 Kayak Trail ~ Norton,Real Estate Agent,40.0,White,No,,2,...,,,,0,empty,1208,<NA>,empty,NaN,<NA>


In [298]:
mark_boston(df, uid)

u_index=971 manually set to 'boston, MA' and confirmed.


,Unnamed: 0,ID Number,Submission Date,Application Property,Source,Age,Race/Ethnicity,Disability,Current Residence,HH Size,...,Current Address,normalized_address,matched_city,matched_state,match_score,match_type,u_index,matched_city_confirmed,match_confirm,matched_state_confirmed
970,590,188403,2024-06-17 22:00:39.000,"33 Intrepid Circle, Unit 305 ~ Marblehead",MyMassHome,48.0,Black or African American,No,Mattapan,2,...,Mattapan,mattapan,mattapoisett,MA,77,no_match,971,boston,manual confirmed,MA


In [434]:
override_city(df, uid, "worcester")
df.loc[df['u_index'] == 1724, 'matched_state_confirmed'] = 'MA'

u_index=1724 manually modified city to 'worcester' and confirmed.


In [392]:
df.loc[df['u_index'] == 1490,
       ['matched_city_confirmed', 'matched_state_confirmed', 'matched_county_confirmed', 'match_confirm']] = ['biddeford', 'ME', 'York County', 'manual confirmed']

In [339]:
df[df['u_index'] == 1156]

,Unnamed: 0,ID Number,Submission Date,Application Property,Source,Age,Race/Ethnicity,Disability,Current Residence,HH Size,...,normalized_address,matched_city,matched_state,match_score,match_type,u_index,matched_city_confirmed,match_confirm,matched_state_confirmed,matched_county_confirmed
1155,775,866897,2024-08-01 18:27:25.000,"401 West Center St, F2, ~ West Bridgewater",Zillow/Trulia/Other Website,30.0,White,No,Fort Smith,5,...,fort smith,north smithfield,RI,81,no_match,1156,fort smith,manual confirmed,AR,Sequoyah County


In [133]:
from shapely.geometry import Point

# --- inputs ---
shp_path = "townssurvey_shp/TOWNSSURVEY_POLY.shp"   # your shapefile
lat, lon = 42.240877, -71.133169          # Brighton point  (lat, lon)

# --- load data ---
gdf = gpd.read_file(shp_path)

# --- point as GeoDataFrame in WGS84 ---
pt = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326")

# --- align CRS and spatial join ---
if gdf.crs is None:
    # if the shapefile somehow has no CRS set, try assuming Mass State Plane Mainland (EPSG:26986)
    gdf = gdf.set_crs("EPSG:26986", allow_override=True)

# reproject the point into the shapefile's CRS for a reliable spatial join
pt = pt.to_crs(gdf.crs)

# primary try: within
hit = gpd.sjoin(pt, gdf[["TOWN", "geometry"]], how="left", predicate="within")

town = hit.iloc[0]["TOWN"]

# if right on a boundary and came back empty, try a tiny buffer (1 meter-ish in projected CRS)
if pd.isna(town):
    hit2 = gpd.sjoin(pt.buffer(1), gdf[["TOWN", "geometry"]], how="left", predicate="intersects")
    town = hit2.iloc[0]["TOWN"]
    
town = str(town).lower()
print("Town/City:", town)

Town/City: boston


Although technically anyone in the U.S. can apply — for example, even residents from Florida — most applicants are from Boston since the housing units are located there. The reason there are so many unmatched cases in the Q&A data is that many Boston residents wrote very specific neighborhood names instead of just listing the city or town. As a result, these detailed neighborhood entries caused many mismatches during the data matching process.

In [445]:
print(df.loc[df["matched_city_confirmed"].isna(), "match_confirm"])

127     empty
166     empty
187     empty
276     empty
355     empty
429     empty
1207    empty
Name: match_confirm, dtype: object


In [460]:
df.to_csv("current_address_cleaned.csv", index=False)

In [100]:
# This part is not connected with the code above
# Update 'matched_state_confirmed'

共有 14 行州不一致（仅比较 Current Residence State 有值的行）：
                Current Residence Current Residence State matched_state_confirmed
                          Chester                      NH                      MA
Moving from providence to Grafton                      RI                      MA
                      Marlborough                      CT                      MA
                       Dorchester                      MA                      NH
                       Dorchester                      MA                      NH
                      charlestown                      RI                      NH
                      charlestown                      RI                      NH
                      Charlestown                      RI                      NH
                           Hudson                      NH                      MA
                           Hudson                      NH                      MA
                          Warwick                  

In [101]:
df2 = pd.read_csv("current_address_cleaned_with_county_2.csv")

assert len(df) == len(df2), f"Row count mismatch: df={len(df)}, df2={len(df2)}"

mask_has_state = df["Current Residence State"].notna() & (df["Current Residence State"].astype(str).str.strip() != "")

mask_diff = mask_has_state & df["Current Residence State"].astype(str).str.strip().str.upper().ne(
    df2["matched_state_confirmed"].astype(str).str.strip().str.upper()
)

df2.loc[mask_diff, "matched_state_confirmed"] = df.loc[mask_diff, "Current Residence State"]

print(f"Replaced {mask_diff.sum()} inconsistent rows in matched_state_confirmed using Current Residence State")

output_path = "current_address_cleaned_with_county_3.csv"
df2.to_csv(output_path, index=False)
print(f"Saved the updated file as: {output_path}")

Replaced 14 inconsistent rows in matched_state_confirmed using Current Residence State
Saved the updated file as: current_address_cleaned_with_county_3.csv
